# Attention Mechanism

## Imports

In [12]:
import numpy as np
import pandas as pd

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Embedding,
    Dense,
    Attention,
    Concatenate,
    Dot,
    Softmax
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

## 1. Data

### 1.1 Prepare data

In [4]:
# Sample data
english_sentences = [
    "i love ai",
    "i love deep learning",
    "how are you",
    "good morning"
]

french_sentences = [
    "start j aime ai end",
    "start j aime apprentissage profond end",
    "start comment allez vous end",
    "start bonjour end"
]

### 1.2 Tokenization

In [5]:
# English
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_sentences)
eng_vocab_size = len(eng_tokenizer.word_index) + 1

# French
fr_tokenizer = Tokenizer()
fr_tokenizer.fit_on_texts(french_sentences)
fr_vocab_size = len(fr_tokenizer.word_index) + 1

# Print vocab sizes
print("English Vocabulary Size:", eng_vocab_size)
print("French Vocabulary Size:", fr_vocab_size)

English Vocabulary Size: 11
French Vocabulary Size: 12


### 1.3 Convert sentences into sequences

In [6]:
# Encoder, decoder inputs
encoder_input = eng_tokenizer.texts_to_sequences(english_sentences)
decoder_input = fr_tokenizer.texts_to_sequences(french_sentences)

# Print sequences
print("English Sequences:", encoder_input)
print("French Sequences:", decoder_input)

English Sequences: [[1, 2, 3], [1, 2, 4, 5], [6, 7, 8], [9, 10]]
French Sequences: [[1, 3, 4, 5, 2], [1, 3, 4, 6, 7, 2], [1, 8, 9, 10, 2], [1, 11, 2]]


### 1.4 Padding

In [8]:
# Pad encoder input sequences
encoder_input = pad_sequences(encoder_input, padding='post')

# Pad decoder input sequences
decoder_input = pad_sequences(decoder_input, padding='post')

# Print padded sequences
print("English Padded Sequences:\n", encoder_input)
print("French Padded Sequences:\n", decoder_input)

English Padded Sequences:
 [[ 1  2  3  0]
 [ 1  2  4  5]
 [ 6  7  8  0]
 [ 9 10  0  0]]
French Padded Sequences:
 [[ 1  3  4  5  2  0]
 [ 1  3  4  6  7  2]
 [ 1  8  9 10  2  0]
 [ 1 11  2  0  0  0]]


## 2. Build Model

### 2.1 Encoder

In [11]:
# Build encoder
encoder_inputs = Input(shape=(None,))
encoder_embedding = Embedding(input_dim=eng_vocab_size,
                              output_dim=265)(encoder_inputs)

encoder_outputs, state_h, state_c = LSTM(units=64,
                                         return_sequences=True,
                                         return_state=True)(encoder_embedding)
print(encoder_outputs.shape) # hidden states for all words
print(state_h.shape) # final hidden state
print(state_c.shape) # final cell state

(None, None, 64)
(None, 64)
(None, 64)


### 2.2 Decoder

In [13]:
# Build decoder
decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(input_dim=fr_vocab_size,
                              output_dim=265)(decoder_inputs)
decoder_lstm = LSTM(units=64,
                    return_sequences=True,
                    return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding)

print(decoder_outputs.shape) # hidden states for all words

(None, None, 64)


### 2.3 Attention Layer
Attention compares:
- Decoder outputs with
- Encoder outputs
and dynamically focuses on important words.

In [15]:
# Attention layer
attention = Attention()
attention_output = attention([decoder_outputs, encoder_outputs])
print(attention_output.shape)

(None, None, 64)


### 2.4 Combine decoder with attention ouputs

In [17]:
decoder_combined = Concatenate(axis=-1)([decoder_outputs, attention_output])
decoder_combined.shape

(None, None, 128)

### 2.5 Prediction layer

In [19]:
output = Dense(
    units=fr_vocab_size,
    activation='softmax'
)(decoder_combined)
output.shape

(None, None, 12)

### 2.6 Model

In [26]:
# Model
model = Model(
    [encoder_inputs, decoder_inputs],
    output
)

# Compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Summary
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 265) │      3,180 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 265) │      2,915 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │     84,480 │ embedding_1[0][0] │
│                     │ 64), (None, 64),  │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, None,     │     84,480 │ embedding[0][0]   │
│                     │ 64), (None, 64),  │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_1         │ (None, None, 64)  │          0 │ lstm_1[0][0],     │
│ (Attention)         │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, None, 128) │          0 │ lstm_1[0][0],     │
│ (Concatenate)       │                   │            │ attention_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None, 12)  │      1,548 │ concatenate_1[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 176,603 (689.86 KB)

 Trainable params: 176,603 (689.86 KB)

 Non-trainable params: 0 (0.00 B)

## 3. Train model

In [22]:
decoder_target = np.expand_dims(decoder_input, -1)
decoder_target.shape

(4, 6, 1)

In [27]:
model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    epochs=20,
    batch_size=2,
    verbose=1
)

Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 50ms/step - accuracy: 0.3750 - loss: 1.6889
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.4583 - loss: 1.6288
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.5417 - loss: 1.5739
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.5417 - loss: 1.5165
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.5833 - loss: 1.4586
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5833 - loss: 1.4035
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.6250 - loss: 1.3499
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.6250 - loss: 1.2941
Epoch 9/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.6250 - loss: 1.2394
Epoch 10/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.6250 - loss: 1.1855
Epoch 11/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7083 - loss: 1.1310
Epoch 12/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7083 - loss: 1.0801
E

In [39]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Concatenate

# Encoder inference model
encoder_model = Model(encoder_inputs, [encoder_outputs, state_h, state_c])

# Decoder inference inputs
decoder_state_input_h = Input(shape=(64,))
decoder_state_input_c = Input(shape=(64,))
encoder_outputs_input = Input(shape=(None, 64))

# Get the original Embedding layer object for the decoder from the trained model
decoder_embedding_layer_obj = model.get_layer('embedding_1') # Corrected: Get layer by name from the full model

# Decoder LSTM layer in inference mode (with initial states)
decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_embedding_layer_obj(decoder_inputs), # Apply the original Embedding layer to the decoder_inputs
    initial_state=[decoder_state_input_h, decoder_state_input_c]
)

# Attention layer in inference mode
attention_output_inf = attention([decoder_outputs_inf, encoder_outputs_input])
decoder_combined_inf = Concatenate(axis=-1)([decoder_outputs_inf, attention_output_inf])

# Get the original Dense layer object for the output from the trained model
output_layer_obj = model.get_layer('dense_1') # Corrected: Get Dense layer by name 'dense_1'
output_tokens_inf = output_layer_obj(decoder_combined_inf)

# Decoder inference model
decoder_model = Model(
    [
        decoder_inputs,
        encoder_outputs_input,
        decoder_state_input_h,
        decoder_state_input_c
    ],
    [
        output_tokens_inf,
        state_h_inf,
        state_c_inf
    ]
)

def predict_sequence(input_sentence):
    # 1. Encode the input sequence to get the encoder's output and states
    input_seq = eng_tokenizer.texts_to_sequences([input_sentence])
    input_seq = pad_sequences(input_seq, maxlen=encoder_input.shape[1], padding='post')

    e_output, e_h, e_c = encoder_model.predict(input_seq, verbose=0)

    # 2. Initialize the decoder's input with the 'start' token
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = fr_tokenizer.word_index['start']

    stop_condition = False
    decoded_sentence = []
    max_decoder_seq_length = decoder_input.shape[1] # Get from pre-padded decoder_input

    # 3. Iteratively decode words until 'end' token is predicted or max length is reached
    print(f"--- Predicting for: \"{input_sentence}\" ---")
    iteration_count = 0
    while not stop_condition:
        iteration_count += 1
        print(f"Iteration {iteration_count}:")
        print(f"  target_seq (input to decoder): {target_seq[0,0]} (word: {fr_tokenizer.index_word.get(target_seq[0,0], 'UNKNOWN')})")

        output_tokens, h, c = decoder_model.predict(
            [target_seq, e_output, e_h, e_c], verbose=0
        )

        # Sample a token (get the most likely word)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = fr_tokenizer.index_word.get(sampled_token_index, 'UNKNOWN')

        print(f"  Predicted token index: {sampled_token_index}, Predicted word: '{sampled_word}'")
        print(f"  Current decoded_sentence: {decoded_sentence}")

        # Check for 'end' token or max length
        if sampled_word == 'end' or len(decoded_sentence) >= max_decoder_seq_length:
            print(f"  Stopping condition met: sampled_word='{sampled_word}' or len(decoded_sentence)={len(decoded_sentence)} >= max_len={max_decoder_seq_length}")
            stop_condition = True
        elif sampled_word != 'start': # Avoid adding 'start' token to output
            print(f"  Appending '{sampled_word}' to decoded_sentence.")
            decoded_sentence.append(sampled_word)
        else:
            print(f"  Skipping word '{sampled_word}' (is 'start' token).")


        # Update the target sequence (next input to the decoder)
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # Update states
        e_h, e_c = h, c

    print(f"Final decoded sentence: {' '.join(decoded_sentence)}")
    return ' '.join(decoded_sentence)

In [38]:
predict_sequence("i love ai")

''